In [ ]:
# !git clone https://github.com/argo28072001/SRG_HW1.git
# !pip install pytorch_lightning
# !pip install thop

In [ ]:
import sys
import os
sys.path.append('/content/SRG_HW1')

data_path = '/content/SRG_HW1/data'
os.makedirs(data_path, exist_ok=True)

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
from torch.utils.data import Dataset, DataLoader
from torchaudio.datasets import SPEECHCOMMANDS
from torch.utils.data import DataLoader
import torch.nn as nn
import torchmetrics
import torch
from thop import profile
import time

from melbanks import LogMelFilterBanks

In [ ]:
# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
class BinarySpeechCommandsDataset(Dataset):
    def __init__(self, root_dir, subset='training'):
        self.dataset = SPEECHCOMMANDS(root=root_dir, download=False)
        self.subset = subset
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        # Filter for only "yes" and "no" samples
        self.data = []
        for i in range(len(self.dataset)):
            waveform, sample_rate, label, speaker_id, utterance_number = self.dataset[i]
            if label in ['yes', 'no']:
                # Check split based on validation_list.txt and testing_list.txt
                file_path = f'{speaker_id}/{utterance_number}_{label}.wav'
                if self._check_split(file_path, subset):
                    self.data.append(i)
    
    def _check_split(self, file_path, subset):
        validation_list = os.path.join(self.dataset._path, 'validation_list.txt')
        testing_list = os.path.join(self.dataset._path, 'testing_list.txt')
        
        with open(validation_list, 'r') as f:
            val_files = set(f.read().splitlines())
        with open(testing_list, 'r') as f:
            test_files = set(f.read().splitlines())
            
        if subset == 'training':
            return file_path not in val_files and file_path not in test_files
        elif subset == 'validation':
            return file_path in val_files
        else:  # testing
            return file_path in test_files
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        dataset_idx = self.data[idx]
        waveform, sample_rate, label, _, _ = self.dataset[dataset_idx]
        # Convert label to binary (0 for "no", 1 for "yes")
        label = 1 if label == "yes" else 0
        # Move tensors to GPU
        waveform = waveform.to(self.device)
        label = torch.tensor(label, device=self.device)
        return waveform, label 

In [ ]:
class SpeechClassifier(pl.LightningModule):
    def __init__(self):
        super().__init__()
        
        # Mel spectrogram layer using our custom implementation
        self.mel_spec = LogMelFilterBanks(
            n_fft=400,
            samplerate=16000,
            hop_length=160,
            n_mels=80,
            power=2.0
        )
        
        # CNN layers
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        
        # Fully connected layers
        self.fc_layers = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
        # Metrics
        self.train_acc = torchmetrics.Accuracy(task='binary')
        self.val_acc = torchmetrics.Accuracy(task='binary')
        self.test_acc = torchmetrics.Accuracy(task='binary')
        
    def forward(self, x):
        # Convert to mel spectrograms using our custom implementation
        x = self.mel_spec(x)  # This already includes log
        # Add channel dimension
        x = x.unsqueeze(1)
        # Pass through CNN
        x = self.conv_layers(x)
        # Pass through FC layers
        x = self.fc_layers(x)
        return x
    
    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = nn.BCELoss()(y_hat, y.float().unsqueeze(1))
        self.train_acc(y_hat, y)
        self.log('train_loss', loss)
        self.log('train_acc', self.train_acc, on_step=False, on_epoch=True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = nn.BCELoss()(y_hat, y.float().unsqueeze(1))
        self.val_acc(y_hat, y)
        self.log('val_loss', loss)
        self.log('val_acc', self.val_acc, on_step=False, on_epoch=True)
        
    def test_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        self.test_acc(y_hat, y)
        self.log('test_acc', self.test_acc)
        
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=0.001)
    
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)
    
    def calculate_flops(self, input_size=(1, 1, 16000)):
        input_tensor = torch.randn(input_size)
        flops, _ = profile(self, inputs=(input_tensor,))
        return flops 

In [ ]:
# Data setup
train_dataset = BinarySpeechCommandsDataset(root_dir='./data', subset='training')
val_dataset = BinarySpeechCommandsDataset(root_dir='./data', subset='validation')
test_dataset = BinarySpeechCommandsDataset(root_dir='./data', subset='testing')

# Use GPU-specific DataLoader settings
train_loader = DataLoader(
    train_dataset, 
    batch_size=32, 
    shuffle=True, 
    num_workers=2,  # Reduced for Colab
    pin_memory=True  # This helps speed up data transfer to GPU
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=32, 
    num_workers=2,
    pin_memory=True
)
test_loader = DataLoader(
    test_dataset, 
    batch_size=32, 
    num_workers=2,
    pin_memory=True
)

In [ ]:
# Model setup
model = SpeechClassifier()

# Callbacks
checkpoint_callback = ModelCheckpoint(
    monitor='val_acc',
    dirpath='checkpoints',
    filename='speech-classifier-{epoch:02d}-{val_acc:.2f}',
    save_top_k=3,
    mode='max',
)

# Logger
logger = TensorBoardLogger('logs', name='speech_classifier')

In [ ]:
# Custom callback for epoch time tracking
class TimeCallback(pl.Callback):
    def on_train_epoch_start(self, trainer, pl_module):
        self.epoch_start_time = time.time()
        
    def on_train_epoch_end(self, trainer, pl_module):
        epoch_time = time.time() - self.epoch_start_time
        trainer.logger.experiment.add_scalar('epoch_time', epoch_time, trainer.current_epoch)

In [ ]:
# Training with explicit GPU settings
trainer = pl.Trainer(
    max_epochs=30,
    callbacks=[checkpoint_callback, TimeCallback()],
    logger=logger,
    accelerator='gpu',  # Explicitly specify GPU
    devices=1,          # Number of GPUs to use
    strategy='auto'     # Let PyTorch Lightning choose the best strategy
)

# Print model statistics
print(f"Number of parameters: {model.count_parameters():,}")
print(f"Estimated FLOPs: {model.calculate_flops():,}")

In [ ]:
# Train the model
trainer.fit(model, train_loader, val_loader)

# Test the model
trainer.test(model, test_loader)